# Week 38, Day 6: Forge: Week 38 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w038/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", 6 / 2, "deg/s")
nr = lambda c, c50: c ** 2 / (c50 ** 2 + c ** 2)
print("2.", nr(0.2, 0.2), round(nr(0.6, 0.2), 2))
ph = np.linspace(0, 2 * np.pi, 256, endpoint=False); F = np.fft.rfft(np.abs(np.cos(ph)))
print("3.", round(2 * abs(F[1]) / abs(F[0]), 3), "(|cos| repeats twice per cycle, so its power is at the second harmonic)")

### Exercise: Part A: simple or complex, and tuning

Using counts[cell, direction, phase_bin], compute for each cell: pref_dir (the direction with the largest mean count), f1f0 at that direction, is_simple (f1f0 above 1), osi from the vector sum at doubled angles of the mean counts, and dsi = (R_pref - R_opposite) / (R_pref + R_opposite).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
r = np.random.default_rng(1992)
n_cells, dirs, nph, reps = 40, np.arange(0, 360, 30), 16, 10
contrasts = np.array([0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0])
true_simple = r.random(n_cells) < 0.5
true_dir = r.uniform(0, 360, n_cells)
true_ds = r.random(n_cells) < 0.4
null_gain = np.where(true_ds, r.uniform(0, 0.2, n_cells), r.uniform(0.7, 1.0, n_cells))
kappa = r.uniform(1.5, 4, n_cells); rmax = r.uniform(8, 25, n_cells); c50 = r.uniform(0.08, 0.4, n_cells)
phase = np.linspace(0, 2 * np.pi, nph, endpoint=False)

def rate(i, d, c):
    th = np.deg2rad(d - true_dir[i])
    g = np.exp(kappa[i] * (np.cos(th) - 1)) + null_gain[i] * np.exp(kappa[i] * (np.cos(th + np.pi) - 1))
    nr = c ** 2 / (c50[i] ** 2 + c ** 2)
    shape = np.maximum(0, np.cos(phase)) * np.pi if true_simple[i] else 1 + 0.25 * np.cos(phase)
    return 0.3 + rmax[i] * g * nr * shape / 4

counts = np.array([[r.poisson(np.tile(rate(i, d, 1.0), (reps, 1))).mean(0) for d in dirs] for i in range(n_cells)])
mean_dir = counts.mean(2)                                   # cells by directions

def f1f0(x):
    F = np.fft.rfft(x)
    return 2 * np.abs(F[1]) / np.abs(F[0])

pref_idx = ...
pref_dir = dirs[pref_idx]
f1 = ...
is_simple = ...
osi = ...
opp = (pref_idx + 6) % 12
dsi = ...
print(f"{is_simple.sum()} cells called simple ({true_simple.sum()} truly simple); median OSI {np.median(osi):.2f}, median DSI {np.median(dsi):.2f}")
plt.hist(f1, 20); plt.axvline(1, color="k", ls="--"); plt.xlabel("F1/F0 at the preferred direction"); plt.show()

In [ ]:
import numpy as np
def test_class():
    "at least 36 of the 40 cells are classified correctly as simple or complex"
    assert np.sum(is_simple == true_simple) >= 36

def test_pref():
    "the preferred direction is within 30 degrees of the truth for at least 36 cells"
    err = np.abs((pref_dir - true_dir + 180) % 360 - 180)
    assert np.sum(err <= 30) >= 36

def test_dsi():
    "cells built to be direction selective have higher DSI than the others on average"
    assert dsi[true_ds].mean() > dsi[~true_ds].mean() + 0.3

test_class()
test_pref()
test_dsi()
print("All checks passed")

### Exercise: Part B: Naka-Rushton fits

Write naka(c, b, rm, c50_, n) and fit it to each cell's contrast response with curve_fit (starting values b = 0.3, rm = the maximum response, c50 = 0.2, n = 2; bounds keep c50 in [0.01, 1] and n in [0.5, 6]). Store the fitted c50 and n for each cell.

In [ ]:
cresp = np.array([[r.poisson(np.tile(rate(i, pref_dir[i], c), (reps, 1))).mean() for c in contrasts] for i in range(n_cells)])

def naka(c, b, rm, c50_, n):
    return ...

c50_fit, n_fit = np.zeros(n_cells), np.zeros(n_cells)
for i in range(n_cells):
    p, _ = curve_fit(naka, contrasts, cresp[i], p0=[0.3, cresp[i].max(), 0.2, 2],
                     bounds=([0, 0, 0.01, 0.5], [5, 100, 1, 6]), maxfev=20000)
    c50_fit[i], n_fit[i] = ...
print(f"correlation of fitted and true c50: {np.corrcoef(c50_fit, c50)[0, 1]:.2f}; median exponent {np.median(n_fit):.2f}")
plt.plot(c50, c50_fit, "o"); plt.plot([0, 0.45], [0, 0.45], "k--"); plt.xlabel("true c50"); plt.ylabel("fitted c50"); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "the fitted c50 correlates with the true c50 above 0.85 across cells"
    assert np.corrcoef(c50_fit, c50)[0, 1] > 0.85

def test_exponent():
    "the median fitted exponent is between 1.5 and 2.5 (the model's true value is 2)"
    assert 1.5 < np.median(n_fit) < 2.5

test_fit()
test_exponent()
print("All checks passed")